# FINAL Methodology Implementation

## Overview
In this notebook we'll conduct the full implementation of our final methodology. The steps are as follows:
1. Load Data
2. Preprocess Data
3. Create Baseline Features
4. Compute NLP Features
5. Pre-Compute TD-Combinations of Dataframes
6. Train-Validation-Test split (70, 20, 10)
7. Fit & Tune Ridge Regression on all TD combinations (only train set) and find top 5
8. Fit & Tune XGBoost on top 5 TD combinations (only train set) and find best one
9. Predict validation set using XGBoost and use that data to fine tune Decision Model
10. Use XGBoost and Decision model on test set

(The above steps need to be refined)


---
## Importing Required Libraries
This section imports all necessary libraries for running the analysis in this notebook

In [ ]:
# Uncomment if this is the first time running the notebook
#!pip install pandas numpy datetime tqdm requests os dotenv umap-learn torch transformers sentence-transformers datasets scikit-learn xgboost joblib matplotlib seaborn

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm

# API libraries
import requests
import os
from dotenv import load_dotenv

# NLP libraries
import umap
import torch
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from datasets import Dataset

# ML Libraries
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, RidgeClassifierCV
from sklearn.base import clone
from xgboost import XGBRegressor, XGBClassifier
import lightgbm as lgb
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Parallelization libraries
from joblib import Parallel, delayed
from scipy import stats

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns

---
# Data Collection & Loading
## News Data
Here we only need to import `german_news_v1.csv` into a dataframe since we already collected and somewhat pre-processed it.

## TODO: IMPLEMENT FULL DATASET

In [ ]:
# CHECK: Times in news_df are timezone aware (UTC+0), switching to timezone naive

news_df = pd.read_csv('german_news_v1.csv')

# convert publishedAt to datetime
news_df['publishedAt'] = pd.to_datetime(news_df['publishedAt'])

# Change datetime to timezone naive
news_df['publishedAt'] = news_df['publishedAt'].dt.tz_localize(None)

# set publishedAt as index
news_df.set_index('publishedAt', inplace=True)

print(news_df.shape)
news_df.head(5)

In [ ]:
# Remove in final run

# Only keep articles from 2025-01-01 onwards
news_df = news_df[news_df.index >= '2025-01-01']

# Remove everything but 500 random articles to keep training light
news_df = news_df.sample(500)

print(news_df.shape)

## Energy Data
import energy data from `energy_data.csv`

## TODO: CHECK WHY NUCLEAR IS EMPTY IN 2025!

In [ ]:
# CAREFUL: Times in energy_df are timezone naive but set at (UTC+1) so we need to shift by 1 hour

energy_df = pd.read_csv('energy_data.csv')

# drop nuclear column since Germany doesn't have nuclear power plants after 2025
energy_df = energy_df.drop(columns=['Nuclear'])

energy_df['Timestamp'] = pd.to_datetime(energy_df['Timestamp'])
energy_df['Timestamp'] = energy_df['Timestamp'] - pd.Timedelta(hours=1)  # shift all data by 1 hour earlier


energy_df.set_index('Timestamp', inplace=True)

# drop na because of the shift
energy_df = energy_df.dropna()

energy_df.tail(5)

In [ ]:
# Remove in Final Run: We keep only data from 01-01-2025 onwards

energy_df = energy_df[energy_df.index >= '2025-01-01 00:00:00']

energy_df.head(5)

---
# Feature Engineering
In this section we'll be computing the different features used in training our baseline and advanced models.

In [ ]:
# Create new column called 'real_spread' based on Spot Price and Day Ahead Auction Price
energy_df['real_spread_abs'] = energy_df['Spot Price'] - energy_df['Day Ahead Auction']

# drop Non-Renewable and Renewable columns for now
energy_df = energy_df.drop(columns=['Non-Renewable', 'Renewable'])

# create target variable [1, 0, -1] based on real_spread
energy_df['spread_target'] = np.where(energy_df['real_spread_abs'] > 0, 1, np.where(energy_df['real_spread_abs'] < 0, -1, 0))


In [ ]:
energy_df.head()

In [ ]:
# TODO: Fine tune these baseline features

# copy energy_df into master_df
master_df = energy_df.copy()

# shift the real_spread_abs & spread_target 24 hours back
master_df['real_spread_abs_shift_24'] = master_df['real_spread_abs'].shift(-24)
master_df['spread_target_shift_24'] = master_df['spread_target'].shift(-24)

# create lagged features
master_df['price_lag_24'] = master_df['Spot Price'].shift(24)
master_df['price_lag_168'] = master_df['Spot Price'].shift(168)
master_df['load_lag_24'] = master_df['Load'].shift(24)
master_df['load_lag_168'] = master_df['Load'].shift(168)

master_df['hour'] = master_df.index.hour
master_df['week_of_year'] = master_df.index.isocalendar().week
master_df['month'] = master_df.index.month
master_df['day_of_week'] = master_df.index.dayofweek
master_df['day_of_year'] = master_df.index.dayofyear

master_df = master_df.dropna()

print(master_df.columns)
master_df.head(25)

## Topic Count Features
First we define 13 topics to classify news headlines into. These will be changed and optimised later down the line. 
2) 

In [ ]:
# TODO: Fine tune these topics

candidate_labels = [
    # Energy consumption
    "der Strom- oder Energieverbrauch steigt deutlich",
    "der Strom- oder Energieverbrauch sinkt deutlich",

    # Energy production / generation availability
    "die Erzeugung oder Verfügbarkeit von Strom oder Energie steigt",
    "die Erzeugung oder Verfügbarkeit von Strom oder Energie sinkt",

    # Commodity prices (gas/coal/oil)
    "die Preise für Erdgas, Kohle oder Öl steigen stark",
    "die Preise für Erdgas, Kohle oder Öl fallen stark",

    # Geopolitik und Versorgung
    "es gibt zunehmende geopolitische Spannungen oder Lieferengpässe bei Energie",
    "die geopolitischen Spannungen und Versorgungsprobleme gehen zurück",

    # Auswirkungen von Wetter in Deutschland auf Strompreise
    "in Deutschland gibt es kaltes Wetter, wenig Wind oder wenig Sonne mit möglichen Auswirkungen auf die Strompreise",
    "in Deutschland gibt es mildes oder warmes Wetter, viel Wind oder viel Sonne mit möglichen Auswirkungen auf die Strompreise",

    # Finanzmärkte
    "an den Finanzmärkten herrscht Instabilität oder Turbulenz",
    "an den Finanzmärkten herrscht Stabilität oder Beruhigung",

    # Catch-all
    "der Text hat keinen Bezug zu Energiepreisen, Wetter oder Finanzmärkten"
]

hypothesis_template = "Dieser Text legt nahe, dass {}."

We use Zero-Shot Classification to try and classify our news into the 13 topics as defined above. Some more work is needed when it comes to pre-processing and hypothesis crafting to ensure not too many articles fall into the catch-all category.

We use `ahajtomar/German_Zeroshot` as a model here, because it is a little bit more lightweight. In future itterations we might want to use `nahiar/zero-shot-classification` or another larger model for better results.

We also first try to classify news on their headline, since it's shorter and more efficient, then run through any articles that fell into the catch-all category again, but using their longer article description.

In [ ]:
# GPU Optimization Settings
import os
# Set tokenizer parallelism to avoid warnings and improve performance
os.environ["TOKENIZERS_PARALLELISM"] = "true"

# Enable torch optimizations for better GPU utilization
torch.backends.cudnn.benchmark = True  # Optimize for consistent input sizes
if torch.cuda.is_available():
    torch.cuda.empty_cache()  # Clear any cached memory

# Check for device availability in order: CUDA -> MPS -> CPU
if torch.cuda.is_available():
    device = 0
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)  # GB
    print(f"CUDA available: Using GPU 0 ({gpu_name})")
    print(f"GPU Memory: {gpu_memory:.1f} GB")
    print(f"Total GPUs: {num_gpus}")
    
    # Optimize batch size based on GPU memory
    # Larger GPUs (>16GB) can handle larger batches
    if gpu_memory >= 24:
        optimal_batch_size = 512
    elif gpu_memory >= 16:
        optimal_batch_size = 384
    elif gpu_memory >= 8:
        optimal_batch_size = 256
    else:
        optimal_batch_size = 128
    print(f"Optimal batch size for GPU: {optimal_batch_size}")
elif torch.backends.mps.is_available():
    device = "mps"
    optimal_batch_size = 256  # MPS typically handles this well
    print("MPS available: Using Apple Silicon GPU")
else:
    device = -1
    optimal_batch_size = 32
    print("No GPU available: Using CPU")

# Initialize pipeline with optimized settings
# Using fp16 (half precision) for faster inference and lower VRAM usage
classifier = pipeline(
    "zero-shot-classification",
    model="Sahajtomar/German_Zeroshot",
    device=device,
    batch_size=optimal_batch_size,
    torch_dtype=torch.float16 if device != -1 and device != "mps" else torch.float32  # fp16 on CUDA only

)

def classify_batch(texts, labels, hypothesis_template, batch_size=None, show_progress=True):
    """Classify a batch of texts using zero-shot classification with optimized GPU processing."""
    # Use optimal batch size if not specified
    if batch_size is None:
        batch_size = optimal_batch_size
    
    valid_texts = []
    valid_indices = []
    
    for idx, text in enumerate(texts):
        if pd.notna(text) and text.strip() != '':
            valid_texts.append(text)
            valid_indices.append(idx)

    if not valid_texts:
        return {}, {}

    classifications_dict = {}
    scores_dict = {}

    if show_progress:
        print(f"Processing {len(valid_texts)} texts with batch_size={batch_size} for efficient GPU utilization...")
    
    # Pass all texts at once - the pipeline will batch them internally
    # This approach eliminates sequential processing and maximizes GPU throughput
    # The pipeline uses the optimized batch_size and fp16 precision for better performance
    results = classifier(
        valid_texts,  # Pass all texts at once - pipeline handles batching
        labels,
        hypothesis_template=hypothesis_template,
        multi_label=False
    )
    
    # Handle both single result (dict) and multiple results (list)
    if isinstance(results, dict):
        results = [results]
    
    # Map results back to original indices
    for i, (idx, result) in enumerate(zip(valid_indices, results)):
        classifications_dict[idx] = result['labels'][0]
        scores_dict[idx] = result['scores'][0]

    # Fill in None for invalid texts
    for idx in range(len(texts)):
        if idx not in classifications_dict:
            classifications_dict[idx] = None
            scores_dict[idx] = 0.0

    return classifications_dict, scores_dict

titles = news_df['title'].tolist()
classifications_dict, scores_dict = classify_batch(
    titles, candidate_labels, hypothesis_template, batch_size=optimal_batch_size, show_progress=True
)

news_df['classification'] = [classifications_dict[i] for i in range(len(news_df))]
news_df['classification_score'] = [scores_dict[i] for i in range(len(news_df))]

other_mask = news_df['classification'] == "other (not related to these energy price drivers)"
num_other = other_mask.sum()

if num_other > 0:
    other_indices = news_df[other_mask].index
    descriptions = news_df.loc[other_indices, 'description'].tolist()
    
    other_classifications_dict, other_scores_dict = classify_batch(
        descriptions, candidate_labels, hypothesis_template, batch_size=optimal_batch_size, show_progress=True
    )
    
    for i, idx in enumerate(other_indices):
        news_df.loc[idx, 'classification'] = other_classifications_dict[i]
        news_df.loc[idx, 'classification_score'] = other_scores_dict[i]

final_other = (news_df['classification'] == "es ist nichts mit Energiepreisen, Wetter oder Finanzmärkten zu tun hat").sum()

print(f"Classification completed: {len(news_df)} articles processed")
print(f"Articles classified as 'other': {final_other} ({final_other/len(news_df)*100:.1f}%)")
print(f"\nClassification distribution:")
print(news_df['classification'].value_counts())
print(f"\nAverage score: {news_df['classification_score'].mean():.3f}")
print(f"Median score: {news_df['classification_score'].median():.3f}")

In [ ]:
news_df.head(5)

## News Embeddings Features
Prior to aggregating topic counts, we will also create the embeddings for each article. We do this, because we want the embedding **with** classification for each article. This is done in case we want to keep or drop the catch-all topic, prior to computing the Time-Decayed weighted average.

In [ ]:
def compute_embeddings(show_progress=True):
    """
    Compute embeddings for news headlines (titles only) using a pre-trained model. 
    The full embeddings should be inserted back into the `news_df` dataframe.
    
    Args:
        show_progress (bool): Whether to show a progress bar for the embedding computation.
    
    Returns:
        None (modifies news_df in place)
    """

    # GPU optimization: Clear cache before starting
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()  # Ensure all previous operations are complete
    
    if torch.cuda.is_available():
        device = "cuda"
        num_gpus = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)  # GB
        print(f"CUDA available: {num_gpus} GPU(s) detected ({gpu_name})")
        print(f"GPU Memory: {gpu_memory:.1f} GB")
        # Note: SentenceTransformer doesn't support DataParallel - will use first GPU
        use_multi_gpu = False  # Changed: Disable DataParallel wrapping
        
        # Optimize batch size based on GPU memory for embeddings
        if gpu_memory >= 24:
            embedding_batch_size = 512
        elif gpu_memory >= 16:
            embedding_batch_size = 384
        elif gpu_memory >= 8:
            embedding_batch_size = 256
        else:
            embedding_batch_size = 128
    elif torch.backends.mps.is_available():
        device = "mps"
        use_multi_gpu = False
        embedding_batch_size = 256  # MPS typically handles this well
        print("MPS available: Using Apple Silicon GPU")
    else:
        device = "cpu"
        use_multi_gpu = False
        embedding_batch_size = 32
        print("No GPU available: Using CPU")
    
    # Initialize model with optimizations
    # SentenceTransformer automatically handles device placement
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)
    
    # Convert to half precision on CUDA for faster inference and lower VRAM usage
    if device == "cuda":
        try:
            # Convert model to half precision (fp16) if supported
            model = model.half()
            print("Model converted to fp16 (half precision) for faster GPU inference")
        except Exception as e:
            print(f"Note: Could not convert to fp16: {e}. Continuing with fp32.")
    
    # REMOVED: DataParallel wrapping - SentenceTransformer doesn't support this pattern
    # The model will use the first GPU by default which should be sufficient with proper batching
    
    # Extract only titles for embedding - optimized with list comprehension
    # Use iloc instead of loc to handle non-sequential indices correctly
    texts = [
        (news_df.iloc[idx]['title'] if pd.notna(news_df.iloc[idx]['title']) else '').strip() 
        if pd.notna(news_df.iloc[idx]['title']) else ''
        for idx in range(len(news_df))
    ]
    
    # Filter out empty texts
    valid_texts = [t for t in texts if t.strip() != '']
    valid_indices = [i for i, t in enumerate(texts) if t.strip() != '']
    
    if len(valid_texts) == 0:
        print("Warning: No valid texts found for embedding!")
        return None
    
    # Compute embeddings - let SentenceTransformer handle batching internally
    # This is more reliable, especially on MPS devices
    batch_size = embedding_batch_size
    print(f"Using batch_size={batch_size} for embedding computation")
    print(f"Computing embeddings for {len(valid_texts)} articles using device: {device}")
    
    try:
        # Let SentenceTransformer handle batching internally - more reliable
        embeddings_array = model.encode(
            valid_texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            show_progress_bar=show_progress,
            normalize_embeddings=False
        )
        
        # Create a full array with NaN for invalid texts
        full_embeddings = np.full((len(texts), embeddings_array.shape[1]), np.nan, dtype=np.float32)
        for idx, valid_idx in enumerate(valid_indices):
            full_embeddings[valid_idx] = embeddings_array[idx]
        
        embeddings_array = full_embeddings
        
    except Exception as e:
        print(f"Error computing embeddings: {e}")
        print(f"Error type: {type(e).__name__}")
        import traceback
        traceback.print_exc()
        
        # Try with smaller batch size as fallback
        print(f"\nTrying with smaller batch size (32)...")
        try:
            embeddings_array = model.encode(
                valid_texts,
                batch_size=32,
                convert_to_numpy=True,
                show_progress_bar=show_progress,
                normalize_embeddings=False
            )
            
            # Create a full array with NaN for invalid texts
            full_embeddings = np.full((len(texts), embeddings_array.shape[1]), np.nan, dtype=np.float32)
            for idx, valid_idx in enumerate(valid_indices):
                full_embeddings[valid_idx] = embeddings_array[idx]
            
            embeddings_array = full_embeddings
            
        except Exception as e2:
            print(f"Error with smaller batch size: {e2}")
            raise
    
    # Clear GPU cache after computation
    if device == "cuda":
        torch.cuda.empty_cache()
    elif device == "mps":
        try:
            torch.mps.empty_cache()
        except AttributeError:
            pass  # MPS might not have empty_cache on older PyTorch versions
    news_df['embedding'] = [emb for emb in embeddings_array]
    
    # Final GPU cleanup
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    print(f"Embeddings computed: shape {embeddings_array.shape}")
    print(f"Embedding dimension: {embeddings_array.shape[1]}")
    if device != "cpu":
        if torch.cuda.is_available():
            memory_used = torch.cuda.max_memory_allocated(0) / (1024**3)
            print(f"Peak GPU memory used: {memory_used:.2f} GB")
    
    return None

compute_embeddings(show_progress=True)

In [ ]:
print(news_df.shape)
print(news_df.head(5))

## Time-Decayed Aggregation for Topics
We create a new DataFrame called td_topics_df with a column for the hourly timestamp and one column for each weighted average count of topics.

The rough idea is to have a list of features for all of the 12 topics, which represents how many articles about a certain topic have been released in the two weeks (336h). But instead of a pure count, we'll also calculate an exponentailly time-decayed average. The function should be as follows:

$$weight = e^{- \lambda * HoursSincePublication}$$

This means an article that was released 100h ago will contribute less to the average than an article released 1h ago. Each hour the average should be calculated anew.

For this implementation we'll set the max `lookback_window` to 336h and `lambda` to 0.05 for a smoother decay. In future implmentations these parameters would be fine-tuned.

In [ ]:
def compute_time_decayed_topic_counts(news_df, master_df, lookback_window=336, decay_lambda=0.05, verbose=True):
    """
    Compute time-decayed weighted counts for each topic.

    For each timestamp in master_df, compute the weighted average count of articles
    published within the lookback_window, using exponential time decay.

    Weight formula: weight = e^(-lambda * hours_since_publication)

    Parameters:
        news_df: DataFrame with datetime index (publishedAt), with columns ['classification', ...]
        master_df: DataFrame with datetime index representing target timestamps
        lookback_window: Number of hours to look back (default: 336h = 2 weeks)
        decay_lambda: Decay rate parameter (default: 0.05)
        verbose: Whether to print progress messages (default: True)

    Returns:
        pd.DataFrame: DataFrame with datetime index and columns for each topic's weighted count
    """

    # Make a copy to avoid modifying original
    td_topic_news_df = news_df.copy()
    # Ensure index is datetime and reset timezone if needed
    if not isinstance(td_topic_news_df.index, pd.DatetimeIndex):
        td_topic_news_df.index = pd.to_datetime(td_topic_news_df.index)
    if td_topic_news_df.index.tz is not None:
        td_topic_news_df.index = td_topic_news_df.index.tz_localize(None)

    # Get unique topics (excluding NaN)
    topics = td_topic_news_df['classification'].dropna().unique()

    # Create output DataFrame with same index as master_df
    td_topics_df = pd.DataFrame(index=master_df.index)

    if verbose:
        print(f"Computing time-decayed counts for {len(td_topics_df)} timestamps and {len(topics)} topics")
        print(f"Lookback window: {lookback_window}h, decay lambda: {decay_lambda}")
        print("Using vectorized computation for improved performance...")

    # Prepare vectorized data
    timestamps = td_topics_df.index.values
    article_times = td_topic_news_df.index.values  # publishedAt as index
    article_topics = td_topic_news_df['classification'].values

    # Pre-filter valid articles (those with valid topics)
    valid_mask = pd.notna(article_topics)
    article_times_valid = article_times[valid_mask]
    article_topics_valid = article_topics[valid_mask]

    # Topic to col idx
    topic_to_idx = {topic: idx for idx, topic in enumerate(topics)}
    weighted_counts_array = np.zeros((len(timestamps), len(topics)))

    if verbose:
        print(f"Processing {len(article_times_valid)} valid articles across {len(timestamps)} timestamps")

    # Main vectorized outer loop
    for i, timestamp in enumerate(tqdm(timestamps, desc="Processing timestamps", leave=True)):
        cutoff_time = timestamp - np.timedelta64(int(lookback_window), 'h')
        time_mask = (article_times_valid >= cutoff_time) & (article_times_valid <= timestamp)
        if not time_mask.any():
            continue
        # Slice relevant articles
        valid_article_times = article_times_valid[time_mask]
        valid_article_topics = article_topics_valid[time_mask]
        # Get hours since publication
        hours_since = (timestamp - valid_article_times).astype('timedelta64[h]').astype(float)
        weights = np.exp(-decay_lambda * hours_since)
        # Accumulate by topic
        for topic in topics:
            topic_mask = valid_article_topics == topic
            weighted_counts_array[i, topic_to_idx[topic]] = np.sum(weights[topic_mask])

    for idx, topic in enumerate(topics):
        td_topics_df[topic] = weighted_counts_array[:, idx]

    return td_topics_df

# Compute time-decayed topic counts
td_topics_df_test = compute_time_decayed_topic_counts(news_df, master_df, lookback_window=336, decay_lambda=0.05)

In [ ]:
print(td_topics_df_test.shape)
td_topics_df_test.head(5)

## Time-Decayed Aggregation for Embeddings
We'll create a new Dataframe called `td_embeddings_df` which will be used as features for the models. 

Similar to what we did in the last section, we want to compute the time-decayed weighted average embedding for every hour. We'll use the same parameter values for `lookback_window` = 336h and `lambda` = 0.05 for the function:

$$weight = e^{- \lambda * HoursSincePublication}$$

One minor difference though, is that once these time-decayed weighted average embeddings have been calculated for every hour, we'll use UMAP to reduce the dimensionality of the embeddings to 20. We'll keep this modular as a function in case we want to disable it and use the full embeddings as features. For now we'll reduce dimensionality and put those into `td_embeddings_df`.

In [ ]:
def compute_time_decayed_embeddings(news_df, master_df, lookback_window=336, decay_lambda=0.05, verbose=True):
    """
    Compute time-decayed weighted average embeddings (OPTIMIZED VERSION).

    For each timestamp in master_df, compute the weighted average embedding of articles
    published within the lookback_window, using exponential time decay.
    Assumes news_df uses 'publishedAt' as its datetime index.

    Weight formula: weight = e^(-lambda * hours_since_publication)

    Parameters:
        news_df: DataFrame with 'publishedAt' as DatetimeIndex and an 'embedding' column
        master_df: DataFrame with datetime index representing target timestamps
        lookback_window: Number of hours to look back (default: 336h = 2 weeks)
        decay_lambda: Decay rate parameter (default: 0.05)
        verbose: Whether to print progress messages (default: True)

    Returns:
        np.ndarray: Array of weighted average embeddings with shape (n_timestamps, embedding_dim)
    """

    # Check that news_df index is datetime and named 'publishedAt'
    if not isinstance(news_df.index, pd.DatetimeIndex):
        raise ValueError("news_df must have a DatetimeIndex")
    if news_df.index.name != 'publishedAt':
        raise ValueError("news_df index must be named 'publishedAt'")

    td_embedding_news_df = news_df.copy()
    # Remove timezone if exists
    if td_embedding_news_df.index.tz is not None:
        td_embedding_news_df.index = td_embedding_news_df.index.tz_localize(None)

    if verbose:
        print(f"Computing time-decayed weighted average embeddings for {len(master_df)} timestamps")
        print(f"Lookback window: {lookback_window}h, decay lambda: {decay_lambda}")
        print("Using optimized vectorized computation...")

    # OPTIMIZATION 1: Vectorized preprocessing of embeddings
    valid_embeddings_mask = td_embedding_news_df['embedding'].notna()
    if not valid_embeddings_mask.any():
        raise ValueError("No valid embeddings found. Check that news_df has valid embeddings.")

    # Get valid indices and process in one go
    valid_indices = td_embedding_news_df[valid_embeddings_mask].index

    # Convert embeddings to numpy array efficiently
    article_embeddings_list = []
    for idx in valid_indices:
        emb = td_embedding_news_df.loc[idx, 'embedding']
        if isinstance(emb, list):
            emb = np.array(emb, dtype=np.float32)  # Use float32 for memory efficiency
        elif not isinstance(emb, np.ndarray):
            emb = np.array(emb, dtype=np.float32)
        elif emb.dtype != np.float32:
            emb = emb.astype(np.float32)
        article_embeddings_list.append(emb)

    # Get embedding dimension from first valid embedding
    embedding_dim = len(article_embeddings_list[0]) if article_embeddings_list else 384

    # Convert to 2D array: shape (n_valid_articles, embedding_dim)
    article_embeddings_array = np.array(article_embeddings_list, dtype=np.float32)
    
    # Use publishedAt from the index (already sorted by time if DatetimeIndex, but we re-sort)
    article_times_valid = valid_indices.values.astype('datetime64[ns]')

    # Sort articles by time for efficient searching
    sort_indices = np.argsort(article_times_valid)
    article_times_valid = article_times_valid[sort_indices]
    article_embeddings_array = article_embeddings_array[sort_indices]

    # Initialize output array: shape (n_timestamps, embedding_dim)
    timestamps = master_df.index.values.astype('datetime64[ns]')
    weighted_embeddings_array = np.zeros((len(timestamps), embedding_dim), dtype=np.float32)

    if verbose:
        print(f"Processing {len(article_embeddings_array)} valid embeddings across {len(timestamps)} timestamps")
        print("Using binary search for efficient article lookup...")

    # OPTIMIZATION 2: Use binary search (searchsorted) to find relevant articles efficiently
    # Convert lookback_window to timedelta64
    lookback_delta = np.timedelta64(int(lookback_window), 'h')

    # Pre-compute cutoff times for all timestamps
    cutoff_times = timestamps - lookback_delta

    # Process timestamps with progress bar
    for i in tqdm(range(len(timestamps)), desc="Processing timestamps", disable=not verbose):
        timestamp = timestamps[i]
        cutoff_time = cutoff_times[i]

        # OPTIMIZATION 3: Use binary search to find articles in range [cutoff_time, timestamp]
        # searchsorted finds insertion points - very fast for sorted arrays
        start_idx = np.searchsorted(article_times_valid, cutoff_time, side='left')
        end_idx = np.searchsorted(article_times_valid, timestamp, side='right')

        if start_idx >= end_idx:
            # No articles in this window
            continue

        # Get articles in the time window
        window_embeddings = article_embeddings_array[start_idx:end_idx]
        window_times = article_times_valid[start_idx:end_idx]

        # OPTIMIZATION 4: Vectorized hours and weights calculation
        hours_since = (timestamp - window_times).astype('timedelta64[h]').astype(np.float32)
        weights = np.exp(-decay_lambda * hours_since, dtype=np.float32)

        # OPTIMIZATION 5: Efficient weighted sum using einsum or broadcasting
        total_weight = np.sum(weights)
        if total_weight > 1e-10:  # Avoid division by very small numbers
            # weights: (n_articles,), window_embeddings: (n_articles, embedding_dim)
            # Result: (embedding_dim,)
            weighted_sum = np.sum(weights[:, np.newaxis] * window_embeddings, axis=0)
            weighted_embeddings_array[i] = weighted_sum / total_weight
        else:
            weighted_embeddings_array[i] = np.zeros(embedding_dim, dtype=np.float32)

    if verbose:
        print(f"\nCompleted time-decayed aggregation")
        print(f"Embedding shape: {weighted_embeddings_array.shape}")

    return weighted_embeddings_array

# Compute time-decayed weighted average embeddings
weighted_embeddings_array_test = compute_time_decayed_embeddings(
    news_df, master_df, lookback_window=336, decay_lambda=0.05
)

In [ ]:
# Apply UMAP dimensionality reduction to the time-decayed embeddings
print(f"Applying UMAP to reduce embeddings from {weighted_embeddings_array_test.shape[1]} to 20 dimensions...")

# Check for NaN values and handle them
nan_mask = np.isnan(weighted_embeddings_array_test).any(axis=1)
num_nan_rows = nan_mask.sum()
print(f"Found {num_nan_rows} rows with NaN values. Filling with zeros before UMAP...")

# Fill NaN values with zeros for UMAP (preserves array shape and index alignment)
weighted_embeddings_clean_test = weighted_embeddings_array_test.copy()
weighted_embeddings_clean_test[nan_mask] = 0.0

# Removed random_state to enable full parallelism with n_jobs=-1
reducer = umap.UMAP(n_components=20, n_jobs=-1, verbose=False)
reduced_embeddings_test = reducer.fit_transform(weighted_embeddings_clean_test)

# Create DataFrame with reduced embeddings
td_embeddings_df_test = pd.DataFrame(
    reduced_embeddings_test,
    index=master_df.index,
    columns=[f'embedding_dim_{i}' for i in range(20)]
)

print(f"Output shape: {td_embeddings_df_test.shape}")


In [ ]:
print(td_embeddings_df_test.shape)
td_embeddings_df_test.head(5)

now we merge all the features together

In [ ]:
# Store the merged_df_test for baseline reference
merged_df_test = master_df.join([td_topics_df_test, td_embeddings_df_test], how='left')
print(merged_df_test.shape)
merged_df_test.head(5)

---
# Grid Search for Time-Decayed Parameters
This section performs a grid search to find optimal time-decayed parameters (`lookback_window` and `decay_lambda`) using Ridge Regression for fast exploration. We'll precompute features for all parameter combinations, then use a simplified expanding window for grid search.


In [ ]:
# TODO: In future runs add more parameters to the grid search

# Define parameter grid for grid search
lookback_windows = [168, 336]  # 1, 2 weeks in hours
decay_lambdas = [0.01, 0.02, 0.05]

# Create all combinations
param_combinations = []
for lw in lookback_windows:
    for dl in decay_lambdas:
        param_combinations.append({'lookback_window': lw, 'decay_lambda': dl})

print(f"Total parameter combinations: {len(param_combinations)}")
print(f"Lookback windows: {lookback_windows}")
print(f"Decay lambdas: {decay_lambdas}")
print(f"\nFirst 3 combinations (showing sample):")
for i, combo in enumerate(param_combinations[:3], 1):
    print(f"{i}. lookback_window={combo['lookback_window']}h, decay_lambda={combo['decay_lambda']}")


In [ ]:
def precompute_single_combination(params):
    """Helper function to precompute features for a single parameter combination."""
    lw = params['lookback_window']
    dl = params['decay_lambda']
    
    # Compute topic counts
    td_topics = compute_time_decayed_topic_counts(news_df, master_df, lookback_window=lw, decay_lambda=dl, verbose=False)
    
    # Compute embeddings
    weighted_embeddings = compute_time_decayed_embeddings(news_df, master_df, lookback_window=lw, decay_lambda=dl, verbose=False)
    
    # Handle NaN values before PCA (PCA doesn't accept NaN)
    nan_mask = np.isnan(weighted_embeddings).any(axis=1)
    weighted_embeddings_clean = weighted_embeddings.copy()
    weighted_embeddings_clean[nan_mask] = 0.0
    
    # Replace UMAP with PCA for faster dimensionality reduction
    reducer = PCA(n_components=20)
    reduced_embeddings = reducer.fit_transform(weighted_embeddings_clean)
    
    td_embeddings = pd.DataFrame(
        reduced_embeddings,
        index=master_df.index,
        columns=[f'embedding_dim_{i}' for i in range(20)]
    )
    
    return (lw, dl), (td_topics, td_embeddings)

In [ ]:
# Precompute features for all parameter combinations
# This will compute time-decayed topics and embeddings for each combination
print(f"Precomputing features for {len(param_combinations)} parameter combinations...")
print("This may take a while, but will speed up the grid search significantly...\n")

# Use parallel processing to precompute features
precomputed_features = dict(
    Parallel(n_jobs=-1, verbose=10)(
        delayed(precompute_single_combination)(params)
        for params in param_combinations
    )
)

print(f"\nPrecomputation complete! Features computed for {len(precomputed_features)} parameter combinations.")
print(f"Sample keys: {list(precomputed_features.keys())[:3]}")

In [ ]:
# Merge and split all precomputed datasets
# This eliminates redundant merging and splitting operations later
print("Merging and splitting all precomputed datasets...")
print("This will create 70% train / 20% validation / 10% test splits for each dataset\n")

preprocessed_datasets = {}
master_df_pre_name_map = {}

for idx, (params_key, (td_topics_df, td_embeddings_df)) in enumerate(precomputed_features.items(), start=1):
    # Step 1: Merge topics and embeddings with master_df ONCE
    merged_features_df = master_df.join([td_topics_df, td_embeddings_df], how='left')
    
    # Step 2: Drop rows with NaN targets ONCE (classification target)
    model_df = merged_features_df.dropna(subset=['spread_target_shift_24']).copy()
    dataset_name = f"master_df_pre_{idx}"
    
    # Persist the full dataframe for this parameter combination
    globals()[dataset_name] = model_df
    master_df_pre_name_map[params_key] = dataset_name
    
    # Step 3: Split into train (70%), validation (20%), test (10%) ONCE
    train_size = int(len(model_df) * 0.7)
    val_size = int(len(model_df) * 0.2)
    train_df = model_df.iloc[:train_size].copy()
    val_df = model_df.iloc[train_size:train_size + val_size].copy()
    test_df = model_df.iloc[train_size + val_size:].copy()
    
    # Store split dataframes as standalone variables for convenience
    globals()[f"{dataset_name}_train"] = train_df
    globals()[f"{dataset_name}_val"] = val_df
    globals()[f"{dataset_name}_test"] = test_df
    
    # Get feature columns for later use
    topic_cols = list(td_topics_df.columns)
    embedding_cols = list(td_embeddings_df.columns)
    news_features = topic_cols + embedding_cols
    
    # Store preprocessed data
    preprocessed_datasets[params_key] = {
        'dataset_name': dataset_name,
        'train_df': train_df,
        'val_df': val_df,
        'test_df': test_df,
        'news_features': news_features,
        'topic_cols': topic_cols,
        'embedding_cols': embedding_cols
    }

print(f"Preprocessing complete! Processed {len(preprocessed_datasets)} datasets.")

if preprocessed_datasets:
    sample_key = next(iter(preprocessed_datasets))
    sample_name = master_df_pre_name_map[sample_key]
    sample_data = preprocessed_datasets[sample_key]
    print(f"Sample dataset: {sample_name}")
    print(f"  Train: {len(sample_data['train_df'])} samples")
    print(f"  Validation: {len(sample_data['val_df'])} samples")
    print(f"  Test: {len(sample_data['test_df'])} samples")


In [ ]:
# Pre-scale all 20 preprocessed datasets
# Fit scaler on training set and transform train/validation/test sets
print("Pre-scaling all 20 datasets...")
print("Fitting scaler on training set and transforming all splits\n")

for params_key, data_dict in preprocessed_datasets.items():
    train_df = data_dict['train_df']
    val_df = data_dict['val_df']
    test_df = data_dict['test_df']
    news_features = data_dict['news_features']
    
    # Fit scaler on training set's news features only
    scaler_news = StandardScaler()
    train_news_features = train_df[news_features].fillna(0)
    scaler_news.fit(train_news_features)
    
    # Transform all splits using the fitted scaler
    # We'll create new columns with '_scaled' suffix for the news features
    for df_name, df in [('train_df', train_df), ('val_df', val_df), ('test_df', test_df)]:
        news_data = df[news_features].fillna(0)
        news_scaled = scaler_news.transform(news_data)
        
        # Create scaled column names
        for idx, feat in enumerate(news_features):
            scaled_col = f"{feat}_scaled"
            df[scaled_col] = news_scaled[:, idx]
        
        # Update the dataframe in the dictionary
        data_dict[df_name] = df
    
    # Store the scaler for later use
    data_dict['scaler_news'] = scaler_news
    
    # Store scaled news feature names
    scaled_news_features = [f"{feat}_scaled" for feat in news_features]
    data_dict['scaled_news_features'] = scaled_news_features

print("Scaling complete! All datasets have been scaled.")
# Get sample scaled feature count from first dataset
sample_key = list(preprocessed_datasets.keys())[0]
sample_scaled_features = preprocessed_datasets[sample_key]['scaled_news_features']
print(f"Scaled news features created: {len(sample_scaled_features)} features per dataset")


In [ ]:
TARGET_COLUMN = 'spread_target_shift_24'
DEFAULT_ALPHAS = np.logspace(-3, 3, 13)


def evaluate_single_parameter_combination(params_key, data_dict, baseline_features,
                                          alphas=None, max_splits=5):
    """Evaluate a single parameter combination using expanding-window RidgeCV confined to the training split."""
    lw, dl = params_key
    dataset_name = data_dict['dataset_name']
    train_df = data_dict['train_df']
    val_df = data_dict['val_df']
    scaled_news_features = data_dict['scaled_news_features']

    feature_columns = baseline_features + scaled_news_features
    missing_features = [col for col in feature_columns if col not in train_df.columns]
    if missing_features:
        raise ValueError(f"Missing features {missing_features} in dataset {dataset_name}")

    if alphas is None:
        alphas = DEFAULT_ALPHAS

    X_train = train_df[feature_columns].fillna(0)
    y_train = train_df[TARGET_COLUMN].astype(int)
    X_val = val_df[feature_columns].fillna(0)
    y_val = val_df[TARGET_COLUMN].astype(int)

    unique_classes = np.unique(y_train)
    if unique_classes.size < 2:
        return {
            'lookback_window': lw,
            'decay_lambda': dl,
            'dataset_name': dataset_name,
            'best_alpha': None,
            'val_accuracy': np.nan,
            'val_macro_f1': np.nan,
            'model': None,
            'params_key': params_key,
            'skip_reason': 'Training split lacks class diversity'
        }

    if len(X_val) == 0:
        return {
            'lookback_window': lw,
            'decay_lambda': dl,
            'dataset_name': dataset_name,
            'best_alpha': None,
            'val_accuracy': np.nan,
            'val_macro_f1': np.nan,
            'model': None,
            'params_key': params_key,
            'skip_reason': 'Validation split is empty'
        }

    max_possible_splits = max(0, len(X_train) - 1)
    effective_splits = min(max_splits, max_possible_splits)

    if effective_splits < 2:
        return {
            'lookback_window': lw,
            'decay_lambda': dl,
            'dataset_name': dataset_name,
            'best_alpha': None,
            'val_accuracy': np.nan,
            'val_macro_f1': np.nan,
            'model': None,
            'params_key': params_key,
            'skip_reason': 'Insufficient data for expanding-window CV'
        }

    tscv = TimeSeriesSplit(n_splits=effective_splits)

    ridge_cv = RidgeClassifierCV(alphas=alphas, cv=tscv, scoring='accuracy')
    ridge_cv.fit(X_train, y_train)

    val_predictions = ridge_cv.predict(X_val)
    val_accuracy = accuracy_score(y_val, val_predictions)
    val_macro_f1 = f1_score(y_val, val_predictions, average='macro', zero_division=0)

    return {
        'lookback_window': lw,
        'decay_lambda': dl,
        'dataset_name': dataset_name,
        'best_alpha': ridge_cv.alpha_,
        'val_accuracy': val_accuracy,
        'val_macro_f1': val_macro_f1,
        'model': ridge_cv,
        'params_key': params_key,
        'skip_reason': None
    }


def grid_search_time_decay_params(preprocessed_datasets, baseline_features,
                                  alphas=None, max_splits=5):
    """Run expanding-window RidgeCV on each precomputed dataset and rank by validation performance."""
    print(f"Grid searching {len(preprocessed_datasets)} parameter combinations...")
    print(f"Using expanding-window RidgeCV confined to training splits (target: {TARGET_COLUMN})")
    print(f"Parallelizing evaluation across parameter combinations using joblib...\n")

    results = Parallel(n_jobs=-1, verbose=10)(
        delayed(evaluate_single_parameter_combination)(
            params_key,
            data_dict,
            baseline_features,
            alphas,
            max_splits
        )
        for params_key, data_dict in preprocessed_datasets.items()
    )

    # Filter out combinations that could not be evaluated
    valid_results = [res for res in results if res['skip_reason'] is None]
    invalid_results = [res for res in results if res['skip_reason'] is not None]

    if invalid_results:
        print("The following combinations were skipped:")
        for res in invalid_results:
            print(f"  - {res['dataset_name']} (lookback={res['lookback_window']}, lambda={res['decay_lambda']}): {res['skip_reason']}")
        print()

    if not valid_results:
        print("No valid results to rank. Please review the dataset or parameters.")
        return []

    # Sort by validation accuracy (descending), then macro F1
    results_sorted = sorted(
        valid_results,
        key=lambda x: (x['val_accuracy'], x['val_macro_f1']),
        reverse=True
    )

    globals()['ridgecv_valid_results'] = valid_results
    globals()['ridgecv_results_sorted'] = results_sorted

    top_results = results_sorted[:3]

    print(f"{'='*80}")
    print("TOP 3 PARAMETER COMBINATIONS:")
    print(f"{'='*80}")
    for idx, result in enumerate(top_results, 1):
        print(
            f"{idx}. dataset={result['dataset_name']} | lookback={result['lookback_window']}h | "
            f"lambda={result['decay_lambda']} | alpha={result['best_alpha']:.4f} | "
            f"Val Accuracy={result['val_accuracy']:.3f} | Val Macro-F1={result['val_macro_f1']:.3f}"
        )

    return top_results


In [ ]:
# Define baseline features (will be used for all combinations)
# Note: Data splitting is now done in the preprocessing step (Cells 33-34)
baseline_features = [
    'price_lag_24', 'price_lag_168',
    'load_lag_24', 'load_lag_168',
    'hour', 'month', 'day_of_week', 'day_of_year', 'week_of_year'
]

print("Baseline features defined:")
print(baseline_features)


In [ ]:
# Run grid search to find top 3 parameter combinations using expanding-window RidgeCV
top_3_combinations = grid_search_time_decay_params(
    preprocessed_datasets=preprocessed_datasets,
    baseline_features=baseline_features,
    alphas=DEFAULT_ALPHAS,
    max_splits=5
)


In [ ]:
# Create a compact summary DataFrame for downstream reporting of the top combinations
if 'top_3_combinations' in globals() and top_3_combinations:
    top_3_summary = pd.DataFrame([
        {
            'dataset_name': res['dataset_name'],
            'lookback_window': res['lookback_window'],
            'decay_lambda': res['decay_lambda'],
            'best_alpha': res['best_alpha'],
            'val_accuracy': res['val_accuracy'],
            'val_macro_f1': res['val_macro_f1']
        }
        for res in top_3_combinations
    ])
    display(top_3_summary)
else:
    print("No top combinations available to summarize.")



In [ ]:
# Inspect feature availability for the top 3 datasets
if 'top_3_combinations' not in globals() or not top_3_combinations:
    raise RuntimeError("top_3_combinations is not defined. Please rerun the RidgeCV selection cells.")

for rank, result in enumerate(top_3_combinations, start=1):
    params_key = result['params_key']
    dataset_name = result['dataset_name']
    data_dict = preprocessed_datasets[params_key]
    train_df = data_dict['train_df']
    val_df = data_dict['val_df']
    test_df = data_dict['test_df']
    scaled_news_features = data_dict['scaled_news_features']

    print(f"#{rank} {dataset_name} -> lookback: {result['lookback_window']}h, decay_lambda: {result['decay_lambda']}")
    print(f"  Train size: {len(train_df)}, Validation size: {len(val_df)}, Test size: {len(test_df)}")
    print(f"  Baseline features present: {all(feat in train_df.columns for feat in baseline_features)}")
    print(f"  Scaled news features count: {len(scaled_news_features)}")
    print(f"  Class distribution (train):\n{train_df[TARGET_COLUMN].value_counts(dropna=False)}\n")


In [ ]:
class ExpandingWindowSplitter:
    """Custom expanding-window splitter with fixed step size for time series CV."""

    def __init__(self, n_splits=8, step_size=12, min_train_size=336):
        if n_splits < 1:
            raise ValueError("n_splits must be at least 1")
        if step_size < 1:
            raise ValueError("step_size must be at least 1")
        if min_train_size < 1:
            raise ValueError("min_train_size must be at least 1")

        self.n_splits = n_splits
        self.step_size = step_size
        self.min_train_size = min_train_size

    def get_n_splits(self, X=None, y=None, groups=None):
        if X is None:
            return self.n_splits
        n_samples = len(X)
        if n_samples <= self.min_train_size:
            return 0
        possible_splits = (n_samples - self.min_train_size) // self.step_size
        return max(0, min(self.n_splits, possible_splits))

    def split(self, X, y=None, groups=None):
        n_samples = len(X)
        effective_splits = self.get_n_splits(X)
        if effective_splits < self.n_splits:
            raise ValueError(
                f"Requested {self.n_splits} splits, but only {effective_splits} are possible with "
                f"{n_samples} samples. Consider reducing n_splits or the step_size/min_train_size."
            )

        train_end = self.min_train_size
        for split_idx in range(self.n_splits):
            test_start = train_end
            test_end = test_start + self.step_size
            if test_end > n_samples:
                raise ValueError(
                    f"Split {split_idx} exceeds available samples ({n_samples}). "
                    "Try reducing n_splits or step_size."
                )

            train_indices = np.arange(0, train_end)
            test_indices = np.arange(test_start, test_end)
            yield train_indices, test_indices

            train_end = test_end


In [ ]:
def detect_optimal_xgb_device(verbose: bool = True) -> dict:
    """Detect available compute for XGBoost and return configuration metadata."""
    device_info = {
        'device': 'cpu',
        'description': 'CPU',
        'tree_method': 'hist',
        'predictor': 'auto',
        'n_jobs': max(1, os.cpu_count() or 1)
    }

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        device_info.update({
            'device': 'cuda',
            'description': f'CUDA GPU ({gpu_name})',
            'tree_method': 'gpu_hist',
            'predictor': 'gpu_predictor',
            'n_jobs': 0  # let XGBoost manage threads; RandomizedSearch handles parallelism
        })
        if verbose:
            print(f"CUDA available: {gpu_name}. Using gpu_hist tree method for XGBoost.")
        return device_info

    mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    if mps_available:
        device_info.update({
            'device': 'mps',
            'description': 'Apple MPS (fallback to CPU for XGBoost)',
            'tree_method': 'hist',
            'predictor': 'auto'
        })
        if verbose:
            print("MPS available: XGBoost will fall back to CPU hist since GPU backend is unsupported.")
        return device_info

    if verbose:
        print("No GPU detected. Using CPU hist tree method for XGBoost.")
    return device_info


In [ ]:
DEFAULT_EXPANDING_SPLITS = 8
DEFAULT_EXPANDING_STEP = 12
DEFAULT_MIN_TRAIN_SIZE = 336  # 2 weeks of hourly observations

XGB_PARAM_DISTRIBUTIONS = {
    'n_estimators': stats.randint(200, 800),
    'max_depth': stats.randint(2, 9),
    'learning_rate': stats.loguniform(0.01, 0.3),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'gamma': stats.uniform(0.0, 5.0),
    'min_child_weight': stats.randint(1, 10),
    'reg_alpha': stats.loguniform(1e-4, 1e1),
    'reg_lambda': stats.loguniform(1e-3, 1e2)
}


def build_xgb_classifier(random_state: int = 42, device_config: dict | None = None) -> XGBClassifier:
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'enable_categorical': False,
        'random_state': random_state
    }

    if device_config is None:
        params.update({'tree_method': 'hist', 'predictor': 'auto', 'n_jobs': -1})
    else:
        params.update({
            'tree_method': device_config.get('tree_method', 'hist'),
            'predictor': device_config.get('predictor', 'auto'),
            'n_jobs': device_config.get('n_jobs', -1)
        })

    return XGBClassifier(**params)


def map_target_to_binary(y: pd.Series) -> np.ndarray:
    unique_values = set(np.unique(y))
    unexpected_values = unique_values - {-1, 0, 1}
    if unexpected_values:
        raise ValueError(f"Unexpected target values encountered: {unexpected_values}")
    return np.where(y > 0, 1, 0)


def run_xgb_random_search(
    data_dict: dict,
    baseline_features: list,
    param_distributions: dict | None = None,
    n_iter: int = 40,
    random_state: int = 42,
    n_splits: int = DEFAULT_EXPANDING_SPLITS,
    step_size: int = DEFAULT_EXPANDING_STEP,
    min_train_size: int = DEFAULT_MIN_TRAIN_SIZE,
    device_config: dict | None = None
):
    if param_distributions is None:
        param_distributions = XGB_PARAM_DISTRIBUTIONS

    train_df = data_dict['train_df']
    scaled_news_features = data_dict['scaled_news_features']
    feature_columns = baseline_features + scaled_news_features
    missing_features = [col for col in feature_columns if col not in train_df.columns]
    if missing_features:
        raise KeyError(f"Missing features in training dataframe: {missing_features}")

    X_train = train_df[feature_columns].fillna(0)
    y_train_raw = train_df[TARGET_COLUMN].astype(int)
    y_train = map_target_to_binary(y_train_raw)

    unique_classes = np.unique(y_train)
    if unique_classes.size < 2:
        raise ValueError("Training data must contain at least two classes for classification.")

    splitter = ExpandingWindowSplitter(
        n_splits=n_splits,
        step_size=step_size,
        min_train_size=min_train_size
    )

    effective_splits = splitter.get_n_splits(X_train)
    if effective_splits < n_splits:
        raise ValueError(
            f"Only {effective_splits} expanding-window splits available. Adjust n_splits, step_size, "
            f"or min_train_size."
        )

    classifier = build_xgb_classifier(random_state=random_state, device_config=device_config)

    if device_config is None:
        search_n_jobs = -1
    elif device_config.get('device') == 'cuda':
        search_n_jobs = 1  # avoid oversubscribing a single GPU across parallel CV folds
    else:
        search_n_jobs = -1

    if device_config and device_config.get('device') == 'cuda':
        print("Running RandomizedSearchCV with CUDA-accelerated XGBoost (serial CV fits).")
    elif device_config and device_config.get('device') == 'mps':
        print("MPS detected: XGBoost uses CPU hist; parallel CV remains enabled.")
    else:
        print("Using CPU-optimised XGBoost with parallel CV fits.")

    search = RandomizedSearchCV(
        estimator=classifier,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring='f1_macro',
        cv=splitter,
        n_jobs=search_n_jobs,
        random_state=random_state,
        verbose=1,
        refit=True,
        return_train_score=True
    )

    search.fit(X_train, y_train)
    return search, feature_columns


In [ ]:
# Run XGBoost tuning for the top 3 datasets using expanding-window CV
xgb_tuning_runs = []
xgb_best_models = {}
xgb_validation_predictions = {}

if 'top_3_combinations' not in globals() or not top_3_combinations:
    raise RuntimeError("top_3_combinations is not defined. Please rerun the RidgeCV selection cells.")

xgb_device_config = detect_optimal_xgb_device(verbose=True)

for rank, result in enumerate(top_3_combinations, start=1):
    params_key = result['params_key']
    dataset_name = result['dataset_name']
    data_dict = preprocessed_datasets[params_key]

    print("=" * 100)
    print(f"Tuning XGBoost for #{rank}: {dataset_name} (lookback={result['lookback_window']}h, decay_lambda={result['decay_lambda']})")
    print(f"Device configuration: {xgb_device_config['description']}")

    try:
        search, feature_columns = run_xgb_random_search(
            data_dict=data_dict,
            baseline_features=baseline_features,
            n_iter=40,
            random_state=42,
            n_splits=DEFAULT_EXPANDING_SPLITS,
            step_size=DEFAULT_EXPANDING_STEP,
            min_train_size=DEFAULT_MIN_TRAIN_SIZE,
            device_config=xgb_device_config
        )
    except ValueError as err:
        print(f"Skipping {dataset_name} due to error: {err}\n")
        continue

    best_estimator = search.best_estimator_

    # Prepare validation data for evaluation
    val_df = data_dict['val_df']
    X_val = val_df[feature_columns].fillna(0)
    y_val_raw = val_df[TARGET_COLUMN].astype(int)
    y_val_binary = map_target_to_binary(y_val_raw)

    if len(np.unique(y_val_binary)) < 2:
        print(f"Validation split for {dataset_name} lacks both classes; skipping evaluation.\n")
        continue

    val_proba = best_estimator.predict_proba(X_val)[:, 1]
    val_pred_binary = (val_proba >= 0.5).astype(int)
    val_pred_signed = np.where(val_pred_binary == 1, 1, -1)

    val_macro_f1 = f1_score(y_val_binary, val_pred_binary, average='macro', zero_division=0)
    val_accuracy = accuracy_score(y_val_binary, val_pred_binary)

    run_summary = {
        'rank': rank,
        'dataset_name': dataset_name,
        'params_key': params_key,
        'lookback_window': result['lookback_window'],
        'decay_lambda': result['decay_lambda'],
        'best_params': search.best_params_,
        'best_cv_macro_f1': search.best_score_,
        'val_macro_f1': val_macro_f1,
        'val_accuracy': val_accuracy,
        'feature_columns': feature_columns,
        'search_object': search,
        'device_config': xgb_device_config
    }

    xgb_tuning_runs.append(run_summary)
    xgb_best_models[dataset_name] = best_estimator
    xgb_validation_predictions[dataset_name] = {
        'y_true_binary': y_val_binary,
        'y_true_signed': np.where(y_val_binary == 1, 1, -1),
        'y_pred_binary': val_pred_binary,
        'y_pred_signed': val_pred_signed,
        'y_pred_proba': val_proba
    }

    print(f"Best CV macro-F1: {search.best_score_:.4f}")
    print(f"Validation macro-F1: {val_macro_f1:.4f} | Validation accuracy: {val_accuracy:.4f}\n")

del rank, result, params_key, dataset_name, data_dict


In [ ]:
# Summarise tuning outcomes and select the best-performing dataset/model
if not xgb_tuning_runs:
    raise RuntimeError("No XGBoost tuning runs were completed. Check previous cells for errors.")

xgb_results_df = pd.DataFrame([
    {
        'rank': run['rank'],
        'dataset_name': run['dataset_name'],
        'lookback_window': run['lookback_window'],
        'decay_lambda': run['decay_lambda'],
        'device': run['device_config']['description'],
        'best_cv_macro_f1': run['best_cv_macro_f1'],
        'val_macro_f1': run['val_macro_f1'],
        'val_accuracy': run['val_accuracy'],
        'best_params': run['best_params']
    }
    for run in xgb_tuning_runs
]).sort_values(by='val_macro_f1', ascending=False).reset_index(drop=True)

display(xgb_results_df)

best_run = xgb_results_df.iloc[0]
best_dataset_name = best_run['dataset_name']
best_model = xgb_best_models[best_dataset_name]
best_run_details = next(run for run in xgb_tuning_runs if run['dataset_name'] == best_dataset_name)

xgb_model_registry = {
    run['dataset_name']: {
        'best_estimator': xgb_best_models[run['dataset_name']],
        'feature_columns': run['feature_columns'],
        'params_key': run['params_key'],
        'lookback_window': run['lookback_window'],
        'decay_lambda': run['decay_lambda'],
        'best_params': run['best_params'],
        'best_cv_macro_f1': run['best_cv_macro_f1'],
        'val_macro_f1': run['val_macro_f1'],
        'val_accuracy': run['val_accuracy'],
        'search_object': run['search_object'],
        'device_config': run['device_config']
    }
    for run in xgb_tuning_runs
}

best_xgb_model_selection = {
    'dataset_name': best_dataset_name,
    'params_key': best_run_details['params_key'],
    'lookback_window': best_run_details['lookback_window'],
    'decay_lambda': best_run_details['decay_lambda'],
    'best_params': best_run_details['best_params'],
    'best_estimator': best_model,
    'feature_columns': best_run_details['feature_columns'],
    'validation_metrics': {
        'macro_f1': best_run_details['val_macro_f1'],
        'accuracy': best_run_details['val_accuracy']
    },
    'cv_macro_f1': best_run_details['best_cv_macro_f1'],
    'device_config': best_run_details['device_config']
}

print("Selected model for downstream forecasting:")
print(best_xgb_model_selection)


## LightGBM Signal Decision Model

We now build a second-stage model that maps XGBoost forecasts and price inputs to long/short/hold trading signals.


In [ ]:
# Prepare features for the LightGBM signal model using best XGBoost forecasts
if 'best_xgb_model_selection' not in globals():
    raise RuntimeError("best_xgb_model_selection is not defined. Please run the XGBoost tuning cells first.")

signal_params_key = best_xgb_model_selection['params_key']
signal_dataset_name = best_xgb_model_selection['dataset_name']

if signal_params_key not in preprocessed_datasets:
    raise KeyError(f"Dataset for params {signal_params_key} not found in preprocessed_datasets.")

signal_data_dict = preprocessed_datasets[signal_params_key]
train_df = signal_data_dict['train_df'].copy()
val_df = signal_data_dict['val_df'].copy()
test_df = signal_data_dict['test_df'].copy()

xgb_feature_columns = best_xgb_model_selection['feature_columns']
xgb_best_estimator = best_xgb_model_selection['best_estimator']
xgb_device_config = best_xgb_model_selection.get('device_config', {'device': 'cpu', 'tree_method': 'hist', 'predictor': 'auto', 'n_jobs': -1})

# Resolve price columns (handle different naming conventions)
def resolve_column(df: pd.DataFrame, candidates: list[str], logical_name: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"None of the candidate columns {candidates} found for '{logical_name}'. Available columns: {list(df.columns)[:20]} ...")

DAY_AHEAD_COL = resolve_column(train_df, ['Day Ahead Auction', 'day_ahead_auction', 'day_ahead_price'], 'day_ahead_price')
SPOT_COL = resolve_column(train_df, ['Spot Price', 'spot_price'], 'spot_price')

# Helper to instantiate fresh XGBoost classifiers with the best parameters
base_xgb_params = xgb_best_estimator.get_params()
base_xgb_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'enable_categorical': False,
    'tree_method': xgb_device_config.get('tree_method', 'hist'),
    'predictor': xgb_device_config.get('predictor', 'auto'),
    'n_jobs': xgb_device_config.get('n_jobs', -1)
})

def create_xgb_clone() -> XGBClassifier:
    sanitized_params = {}
    for key, value in base_xgb_params.items():
        if isinstance(value, (np.floating, np.float32, np.float64)):
            sanitized_params[key] = float(value)
        elif isinstance(value, (np.integer, np.int32, np.int64)):
            sanitized_params[key] = int(value)
        else:
            sanitized_params[key] = value
    model = XGBClassifier(**sanitized_params)
    return model

# Compute out-of-fold probabilities on the training split (binary target for XGBoost)
X_train_full = train_df[xgb_feature_columns].fillna(0)
y_train_signed = train_df[TARGET_COLUMN].astype(int)
y_train_binary = map_target_to_binary(y_train_signed)

splitter = ExpandingWindowSplitter(
    n_splits=DEFAULT_EXPANDING_SPLITS,
    step_size=DEFAULT_EXPANDING_STEP,
    min_train_size=DEFAULT_MIN_TRAIN_SIZE
)

expected_splits = splitter.get_n_splits(X_train_full)
if expected_splits < DEFAULT_EXPANDING_SPLITS:
    raise ValueError(
        f"Not enough data for the configured expanding-window splits ({expected_splits} < {DEFAULT_EXPANDING_SPLITS})."
    )

oof_probas = np.full(len(X_train_full), np.nan, dtype=float)
oof_fold_ids = np.full(len(X_train_full), -1, dtype=int)

for fold_idx, (tr_idx, va_idx) in enumerate(splitter.split(X_train_full), start=1):
    xgb_model = create_xgb_clone()
    xgb_model.fit(X_train_full.iloc[tr_idx], y_train_binary[tr_idx])
    fold_proba = xgb_model.predict_proba(X_train_full.iloc[va_idx])[:, 1]
    oof_probas[va_idx] = fold_proba
    oof_fold_ids[va_idx] = fold_idx

# Fill any remaining gaps (e.g., tail samples) with predictions from the refit estimator
missing_mask = np.isnan(oof_probas)
if missing_mask.any():
    backup_proba = xgb_best_estimator.predict_proba(X_train_full.iloc[missing_mask])[:, 1]
    oof_probas[missing_mask] = backup_proba
    oof_fold_ids[missing_mask] = 0  # mark as filled from refit model

train_signal_features = pd.DataFrame({
    'xgb_proba': oof_probas,
    'xgb_confidence': np.abs(oof_probas - 0.5) * 2,
    'day_ahead_price': train_df[DAY_AHEAD_COL].values,
    'spot_price': train_df[SPOT_COL].values,
}, index=train_df.index)
train_signal_target = y_train_signed

# Prepare validation features using the refit estimator (trained on full training split)
X_val = val_df[xgb_feature_columns].fillna(0)
y_val_signed = val_df[TARGET_COLUMN].astype(int)
val_probas = xgb_best_estimator.predict_proba(X_val)[:, 1]
val_signal_features = pd.DataFrame({
    'xgb_proba': val_probas,
    'xgb_confidence': np.abs(val_probas - 0.5) * 2,
    'day_ahead_price': val_df[DAY_AHEAD_COL].values,
    'spot_price': val_df[SPOT_COL].values,
}, index=val_df.index)
val_signal_target = y_val_signed

# Fit a fresh XGBoost model on train+validation to generate test-set forecasts
X_full = pd.concat([X_train_full, X_val], axis=0)
y_full_binary = map_target_to_binary(pd.concat([train_signal_target, val_signal_target], axis=0))

xgb_full_model = create_xgb_clone()
xgb_full_model.fit(X_full, y_full_binary)

X_test = test_df[xgb_feature_columns].fillna(0)
y_test_signed = test_df[TARGET_COLUMN].astype(int)
test_probas = xgb_full_model.predict_proba(X_test)[:, 1]

test_signal_features = pd.DataFrame({
    'xgb_proba': test_probas,
    'xgb_confidence': np.abs(test_probas - 0.5) * 2,
    'day_ahead_price': test_df[DAY_AHEAD_COL].values,
    'spot_price': test_df[SPOT_COL].values,
}, index=test_df.index)
test_signal_target = y_test_signed

print(f"Prepared signal datasets for '{signal_dataset_name}' with features: {list(train_signal_features.columns)}")
print(f"Train samples: {len(train_signal_features)}, Validation samples: {len(val_signal_features)}, Test samples: {len(test_signal_features)}")


In [ ]:
def detect_optimal_lgbm_device(verbose: bool = True) -> dict:
    """Return LightGBM device configuration prioritising CUDA, then MPS (fallback to CPU), then CPU."""
    device_info = {
        'device': 'cpu',
        'description': 'CPU',
        'n_jobs': max(1, os.cpu_count() or 1)
    }

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        device_info.update({
            'device': 'gpu',
            'description': f'CUDA GPU ({gpu_name})',
            'gpu_platform_id': 0,
            'gpu_device_id': 0,
            'n_jobs': 0  # LightGBM GPU uses internal parallelism
        })
        if verbose:
            print(f"CUDA available: {gpu_name}. LightGBM will train on GPU.")
        return device_info

    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device_info.update({
            'device': 'cpu',  # LightGBM has no native MPS backend; fall back to CPU
            'description': 'Apple MPS detected (LightGBM falls back to CPU)',
            'n_jobs': max(1, os.cpu_count() or 1)
        })
        if verbose:
            print("MPS available: LightGBM does not support MPS backend directly, defaulting to CPU.")
        return device_info

    if verbose:
        print("No GPU detected. Using CPU for LightGBM.")
    return device_info

lgbm_device_config = detect_optimal_lgbm_device(verbose=True)
print(lgbm_device_config)


In [ ]:
# Train LightGBM decision model with GridSearchCV on OOF-augmented features
label_encoder = LabelEncoder()
label_encoder.fit([-1, 0, 1])  # ensure consistent mapping across all splits

y_train_encoded = label_encoder.transform(train_signal_target)

def build_lgbm_classifier(random_state: int = 42) -> lgb.LGBMClassifier:
    params = {
        'objective': 'multiclass',
        'num_class': len(label_encoder.classes_),
        'boosting_type': 'gbdt',
        'random_state': random_state,
        'verbosity': -1,
        'n_jobs': lgbm_device_config.get('n_jobs', -1)
    }
    if lgbm_device_config.get('device') == 'gpu':
        params.update({
            'device': 'gpu',
            'gpu_platform_id': lgbm_device_config.get('gpu_platform_id', 0),
            'gpu_device_id': lgbm_device_config.get('gpu_device_id', 0)
        })
    else:
        params['device'] = 'cpu'
    return lgb.LGBMClassifier(**params)

lgbm_param_grid = {
    'num_leaves': [31, 63],
    'max_depth': [-1, 8],
    'learning_rate': [0.02, 0.05],
    'n_estimators': [200, 400],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

available_splits = splitter.get_n_splits(train_signal_features)
cv_splits = min(5, available_splits)
if cv_splits < 2:
    raise ValueError("Insufficient data for LightGBM cross-validation. Try reducing min_train_size or step size.")

lgbm_cv = ExpandingWindowSplitter(
    n_splits=cv_splits,
    step_size=DEFAULT_EXPANDING_STEP,
    min_train_size=DEFAULT_MIN_TRAIN_SIZE
)

signal_lgbm_estimator = build_lgbm_classifier(random_state=42)

if lgbm_device_config.get('device') == 'gpu':
    grid_n_jobs = 1  # avoid oversubscribing GPU
else:
    grid_n_jobs = lgbm_device_config.get('n_jobs', -1)

signal_lgbm_grid = GridSearchCV(
    estimator=signal_lgbm_estimator,
    param_grid=lgbm_param_grid,
    scoring='f1_macro',
    cv=lgbm_cv,
    n_jobs=grid_n_jobs,
    verbose=1,
    refit=True,
    return_train_score=True
)

signal_lgbm_grid.fit(train_signal_features, y_train_encoded)

signal_best_lgbm = signal_lgbm_grid.best_estimator_
print("Best LightGBM params:", signal_lgbm_grid.best_params_)
print(f"Best CV macro-F1: {signal_lgbm_grid.best_score_:.4f}")

signal_model_artifacts = {
    'label_encoder': label_encoder,
    'grid_search': signal_lgbm_grid,
    'best_estimator': signal_best_lgbm,
    'feature_names': list(train_signal_features.columns),
    'device_config': lgbm_device_config
}


In [ ]:
# Train baseline LightGBM without news-driven features (price-only inputs)
baseline_feature_cols = ['day_ahead_price', 'spot_price']

train_baseline_features = train_signal_features[baseline_feature_cols].copy()
val_baseline_features = val_signal_features[baseline_feature_cols].copy()
test_baseline_features = test_signal_features[baseline_feature_cols].copy()

y_train_baseline = label_encoder.transform(train_signal_target)

baseline_lgbm_estimator = build_lgbm_classifier(random_state=42)

baseline_param_grid = {
    'num_leaves': [15, 31, 63],
    'max_depth': [-1, 8],
    'learning_rate': [0.02, 0.05, 0.1],
    'n_estimators': [200, 400],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

baseline_cv_splits = min(5, available_splits)
baseline_cv = ExpandingWindowSplitter(
    n_splits=baseline_cv_splits,
    step_size=DEFAULT_EXPANDING_STEP,
    min_train_size=DEFAULT_MIN_TRAIN_SIZE
)

baseline_grid = GridSearchCV(
    estimator=baseline_lgbm_estimator,
    param_grid=baseline_param_grid,
    scoring='f1_macro',
    cv=baseline_cv,
    n_jobs=grid_n_jobs,
    verbose=1,
    refit=True,
    return_train_score=True
)

baseline_grid.fit(train_baseline_features, y_train_baseline)

baseline_best_lgbm = baseline_grid.best_estimator_
print("Baseline LightGBM params:", baseline_grid.best_params_)
print(f"Baseline CV macro-F1: {baseline_grid.best_score_:.4f}")

baseline_model_artifacts = {
    'grid_search': baseline_grid,
    'best_estimator': baseline_best_lgbm,
    'feature_names': baseline_feature_cols,
    'device_config': lgbm_device_config
}


In [ ]:
# Evaluate tuned decision model vs baseline and naive strategy on the test set
y_test_encoded = label_encoder.transform(test_signal_target)

signal_test_pred = signal_best_lgbm.predict(test_signal_features)
signal_test_proba = signal_best_lgbm.predict_proba(test_signal_features)

baseline_test_pred = baseline_best_lgbm.predict(test_baseline_features)
baseline_test_proba = baseline_best_lgbm.predict_proba(test_baseline_features)


def _safe_multiclass_auc(y_true: np.ndarray, proba: np.ndarray) -> float:
    unique_classes = np.unique(y_true)
    if unique_classes.size <= 1:
        return np.nan
    if unique_classes.size == 2:
        # Reduce to binary ROC AUC using the higher label as the positive class
        positive_class = unique_classes.max()
        proba_binary = proba[:, positive_class]
        y_binary = (y_true == positive_class).astype(int)
        return roc_auc_score(y_binary, proba_binary)
    proba_aligned = proba[:, unique_classes]
    return roc_auc_score(
        y_true,
        proba_aligned,
        multi_class='ovo',
        average='macro',
        labels=unique_classes
    )

# Classification metrics
signal_metrics = {
    'Model': 'LightGBM Signal (with news)',
    'Accuracy': accuracy_score(y_test_encoded, signal_test_pred),
    'Macro-F1': f1_score(y_test_encoded, signal_test_pred, average='macro', zero_division=0),
    'AUC (macro)': _safe_multiclass_auc(y_test_encoded, signal_test_proba)
}

baseline_metrics = {
    'Model': 'LightGBM Baseline (price-only)',
    'Accuracy': accuracy_score(y_test_encoded, baseline_test_pred),
    'Macro-F1': f1_score(y_test_encoded, baseline_test_pred, average='macro', zero_division=0),
    'AUC (macro)': _safe_multiclass_auc(y_test_encoded, baseline_test_proba)
}

metrics_df = pd.DataFrame([signal_metrics, baseline_metrics]).set_index('Model')
display(metrics_df)

auc_series = metrics_df['AUC (macro)'].dropna()
observed_labels = np.unique(y_test_encoded)
if observed_labels.size >= 2 and not auc_series.empty:
    from sklearn.preprocessing import label_binarize
    from sklearn.metrics import roc_curve

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=1, label='Chance')

    model_curves = [
        (signal_test_proba, 'LightGBM Signal (with news)', '#4C72B0'),
        (baseline_test_proba, 'LightGBM Baseline (price-only)', '#55A868')
    ]

    if observed_labels.size == 2:
        positive_class = observed_labels.max()
        y_binary = (y_test_encoded == positive_class).astype(int)
        for proba, label, color in model_curves:
            fpr, tpr, _ = roc_curve(y_binary, proba[:, positive_class])
            ax.plot(fpr, tpr, label=f'{label}', color=color, linewidth=2)
    else:
        y_binarised = label_binarize(y_test_encoded, classes=observed_labels)
        for proba, label, color in model_curves:
            for idx, cls in enumerate(observed_labels):
                fpr, tpr, _ = roc_curve(y_binarised[:, idx], proba[:, cls])
                ax.plot(fpr, tpr, label=f'{label} vs class {label_encoder.inverse_transform([cls])[0]}', color=color, linewidth=1.5, alpha=0.6)

    ax.set_title('ROC Curves (Macro AUC shown in table)')
    ax.set_xlabel('False Positive Rate (FPR)')
    ax.set_ylabel('True Positive Rate (TPR)')
    ax.grid(True, alpha=0.2)
    ax.legend(loc='lower right', fontsize='small')
    fig.tight_layout()
    plt.show()

print("\nClassification report (signal model):")
print(
    classification_report(
        y_test_encoded,
        signal_test_pred,
        labels=observed_labels,
        target_names=[str(cls) for cls in label_encoder.inverse_transform(observed_labels)]
    )
)

# Trading performance metrics
spot_series = test_df[SPOT_COL]
day_ahead_series = test_df[DAY_AHEAD_COL]
spread_series = spot_series - day_ahead_series

signal_actions = label_encoder.inverse_transform(signal_test_pred)
baseline_actions = label_encoder.inverse_transform(baseline_test_pred)
naive_actions = np.ones_like(signal_actions, dtype=int)


def actions_to_returns(actions: np.ndarray, spread: pd.Series) -> pd.Series:
    returns = np.where(actions == 1, spread, np.where(actions == -1, -spread, 0.0))
    return pd.Series(returns, index=spread.index)

signal_returns = actions_to_returns(signal_actions, spread_series)
baseline_returns = actions_to_returns(baseline_actions, spread_series)
naive_returns = actions_to_returns(naive_actions, spread_series)


def summarise_returns(returns: pd.Series, label: str) -> dict:
    cumulative = returns.cumsum()
    drawdown = cumulative - cumulative.cummax()
    mean_return = returns.mean()
    std_return = returns.std(ddof=1)
    periods_per_year = 24 * 365  # assume hourly cadence
    sharpe = (mean_return / std_return * np.sqrt(periods_per_year)) if std_return > 0 else np.nan
    return {
        'Strategy': label,
        'Total Return': cumulative.iloc[-1],
        'Average Return': mean_return,
        'Volatility': std_return,
        'Sharpe (annualised)': sharpe,
        'Max Drawdown': drawdown.min()
    }

returns_summary = pd.DataFrame([
    summarise_returns(signal_returns, 'LightGBM Signal'),
    summarise_returns(baseline_returns, 'LightGBM Baseline'),
    summarise_returns(naive_returns, 'Naive Buy-DA/Sell-Spot')
]).set_index('Strategy')

display(returns_summary)

# Plot cumulative returns comparison

def _prepare_plot_series(returns: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    cumulative = returns.cumsum()
    index = cumulative.index
    if isinstance(index, pd.DatetimeIndex):
        if index.tz is not None:
            index = index.tz_convert(None)
        x_values = index.to_pydatetime()
    else:
        x_values = np.arange(len(cumulative))
    return x_values, cumulative.to_numpy()

signal_x, signal_y = _prepare_plot_series(signal_returns)
baseline_x, baseline_y = _prepare_plot_series(baseline_returns)
naive_x, naive_y = _prepare_plot_series(naive_returns)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(signal_x, signal_y, '-', label='LightGBM Signal (with news)', linewidth=2)
ax.plot(baseline_x, baseline_y, '--', label='LightGBM Baseline (price-only)')
ax.plot(naive_x, naive_y, ':', label='Naive Buy-DA/Sell-Spot')
ax.set_title('Cumulative Strategy Returns on Test Set')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Cumulative Return (currency units)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

# Store artifacts for downstream analysis
evaluation_artifacts = {
    'classification_metrics': metrics_df,
    'returns_summary': returns_summary,
    'signal_returns': signal_returns,
    'baseline_returns': baseline_returns,
    'naive_returns': naive_returns,
    'signal_actions': signal_actions,
    'baseline_actions': baseline_actions,
    'naive_actions': naive_actions
}
